### **Problem Statement:**
_A SaaS company is experiencing significant drop-offs between free trial signup and paid conversion, leading to lost revenue opportunities. The objective is to identify early behavioral signals and user segments that indicate low conversion likelihood, and recommend targeted interventions to improve trial-to-paid conversion rates._

##### _Assumptions_
- The free trial is assumed to be exactly 14 days long.
- If a user pays for a subscription right after their 14-day free trial ends, they are success(1). If they don't buy the subscription and stop using the app, they are drop-off(0).
- To find "early warning signals", we will look at what user did during their first 7 days on the platform.
- Any row in feature_usage.csv where error_count > 0 is assumed to represent a user-facing glitch or friction point that negatively impacts user experience.
- A first_response_time_minutes of greater than 120 minutes (2 hours) or a satisfaction_score of 1 or 2 stars during the trial is assumed to be a primary driver of conversion failure.


##### _Data Preparation_

In [21]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [ ]:
# Database Connection & Row Counts

DB_USER = 'root'
DB_PW = 'your_password'
DB_HOST = 'localhost'
DB_PORT = '3306'
DB_NAME = 'saas_conversion'

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PW}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

try:
    tables = pd.read_sql_query("SHOW TABLES", engine)
    print("Tables found in database")
    print(tables)

except Exception as e:
    print(f"Connection Failed: {e}")

for table_name in tables.iloc[:, 0]:
    count_df = pd.read_sql_query(f"SELECT COUNT(*) as total FROM {table_name}", engine)
    total_rows = count_df['total'][0]
    preview_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", engine)
    print("-" * 50)
    print(f"Table: {table_name}")
    print(f"Total Rows: {total_rows}")
    display(preview_df)
    print("\n")

Tables found in database
    Tables_in_saas_conversion
0         ravenstack_accounts
1     ravenstack_churn_events
2    ravenstack_feature_usage
3    ravenstack_subscriptions
4  ravenstack_support_tickets
--------------------------------------------------
Table: ravenstack_accounts
Total Rows: 500


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,0,0
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,0,1
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,0,0
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,1,0
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,0,1




--------------------------------------------------
Table: ravenstack_churn_events
Total Rows: 600


,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,0,0,0,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,1,0,0,None
2,C-a174be,A-b07346,2024-11-12,budget,0.00,0,0,0,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,0,0,0,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,0,1,1,too expensive




--------------------------------------------------
Table: ravenstack_feature_usage
Total Rows: 25000


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,0
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,0
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,0
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,0
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,0




--------------------------------------------------
Table: ravenstack_subscriptions
Total Rows: 5000


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,0,0,0,1,monthly,1
1,S-0f6f44,A-9b9fe9,2024-06-11,None,Pro,17,833,9996,0,0,0,0,monthly,1
2,S-51c0d1,A-659280,2024-11-25,None,Enterprise,62,0,0,1,1,0,0,annual,0
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,0,0,0,1,monthly,1
4,S-cff5a2,A-ba6516,2024-01-10,None,Enterprise,27,5373,64476,0,0,0,0,monthly,1




--------------------------------------------------
Table: ravenstack_support_tickets
Total Rows: 2000


,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,0
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,0
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,0
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,0
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,0


In [23]:
# Filtering important columns
df_accounts = pd.read_sql("""
    SELECT 
        account_id, 
        industry, 
        signup_date, 
        referral_source, 
        plan_tier, 
        seats, 
        is_trial
    FROM ravenstack_accounts
""", engine)

df_subscriptions = pd.read_sql("""
    SELECT 
        subscription_id, 
        account_id, 
        start_date, 
        plan_tier, 
        mrr_amount, 
        is_trial, 
        billing_frequency
    FROM ravenstack_subscriptions
""", engine)

df_feature_usage = pd.read_sql("""
    SELECT 
        subscription_id, 
        usage_date, 
        feature_name, 
        usage_count, 
        usage_duration_secs, 
        error_count,
        is_beta_feature
    FROM ravenstack_feature_usage
""", engine)

df_support_tickets = pd.read_sql("""
    SELECT 
        account_id, 
        submitted_at, 
        resolution_time_hours, 
        first_response_time_minutes, 
        satisfaction_score
    FROM ravenstack_support_tickets
""", engine)

##### 📁 **Dataset Descriptions (RavenStack SaaS)**

This project uses a relational dataset containing 5 tables that simulate a real-world B2B software platform. Below is a simple breakdown of what each table tracks and how it helps us analyze free-trial conversions:

##### 1. Accounts Table (`df_accounts`)
* The master list of companies that signed up for our platform. 
* It tells us **who** the customer is (their industry, company size via 'seats') and **when** they started their 14-day free trial (`signup_date`).

##### 2. Subscriptions Table (`df_subscriptions`)
* The billing history and contract logs for every account.
* This is our **"Answer Key."** By tracking whether an account moves from a trial status to a paid monthly plan (`mrr_amount > 0`), we can determine exactly who converted and who dropped off.

##### 3. Feature Usage Table (`df_feature_usage`)
* The daily activity tracker that logs what users actually do inside the software.
* This is where our **"Early Behavioral Signals"** live. It records how many minutes users spend on specific features and whether they run into annoying technical errors during their trial.

##### 4. Support Tickets Table (`df_support_tickets`)
* The customer service helpdesk logs.
* This helps us measure **onboarding friction**. It tracks if a user got stuck, how fast our team replied to them, and if the user ignored or filled out the satisfaction survey.

##### 5. Churn Events Table (`df_churn_events`) *(Optional/Dropped)*
* Exit surveys and refund records for users who formally cancel.
* Since most free-trial drop-offs simply "ghost" the app without formally canceling, we are bypassing this table to focus purely on active trial-to-paid conversions.

##### _Data Cleaning_

In [24]:
df_accounts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   account_id       500 non-null    object
 1   industry         500 non-null    object
 2   signup_date      500 non-null    object
 3   referral_source  500 non-null    object
 4   plan_tier        500 non-null    object
 5   seats            500 non-null    int64 
 6   is_trial         500 non-null    int64 
dtypes: int64(2), object(5)
memory usage: 27.5+ KB


In [25]:
df_subscriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   subscription_id    5000 non-null   object
 1   account_id         5000 non-null   object
 2   start_date         5000 non-null   object
 3   plan_tier          5000 non-null   object
 4   mrr_amount         5000 non-null   int64 
 5   is_trial           5000 non-null   int64 
 6   billing_frequency  5000 non-null   object
dtypes: int64(2), object(5)
memory usage: 273.6+ KB


In [26]:
df_feature_usage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   subscription_id      25000 non-null  object
 1   usage_date           25000 non-null  object
 2   feature_name         25000 non-null  object
 3   usage_count          25000 non-null  int64 
 4   usage_duration_secs  25000 non-null  int64 
 5   error_count          25000 non-null  int64 
 6   is_beta_feature      25000 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 1.3+ MB


In [27]:
df_support_tickets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   account_id                   2000 non-null   object 
 1   submitted_at                 2000 non-null   object 
 2   resolution_time_hours        2000 non-null   float64
 3   first_response_time_minutes  2000 non-null   int64  
 4   satisfaction_score           1175 non-null   float64
dtypes: float64(2), int64(1), object(2)
memory usage: 78.3+ KB


In [28]:
# Fixing data types
df_accounts['signup_date'] = pd.to_datetime(df_accounts['signup_date'])
df_subscriptions['start_date'] = pd.to_datetime(df_subscriptions['start_date'])
df_feature_usage['usage_date'] = pd.to_datetime(df_feature_usage['usage_date'])
df_support_tickets['submitted_at'] = pd.to_datetime(df_support_tickets['submitted_at'])

In [29]:
# Duplicates Handling
print(df_accounts.duplicated().sum(), df_subscriptions.duplicated().sum(), df_feature_usage.duplicated().sum(), df_support_tickets.duplicated().sum(), sep=",")

0,0,0,0


In [30]:
# Stripping white spaces in text
dataframes = [df_accounts, df_subscriptions, df_feature_usage, df_support_tickets]
for df in dataframes:
    string_cols = df.select_dtypes(include='object').columns
    for col in string_cols:
        df.loc[:, col] = df[col].astype(str).str.strip()

In [31]:
# Checking logical inconsistencies

# Usage before signup
usage_check = df_feature_usage.merge(df_subscriptions[['subscription_id', 'account_id']], on='subscription_id', how='left')
usage_check = usage_check.merge(df_accounts[['account_id', 'signup_date']], on='account_id', how='left')

usage_error = usage_check[usage_check['usage_date'] < usage_check['signup_date']]
print(f"Total rows where usage before signup: {len(usage_error)}")

# Support ticket before signup
ticket_check = df_support_tickets.merge(df_accounts[['account_id', 'signup_date']], on='account_id', how='left')
ticket_error = ticket_check[ticket_check['submitted_at'] < ticket_check['signup_date']]
print(f"Tickets submitted before signup: {len(ticket_error)}")

# Negative engagement metrics
negative_counts = df_feature_usage[(df_feature_usage['usage_count'] < 0) | 
                                   (df_feature_usage['usage_duration_secs'] < 0) | 
                                   (df_feature_usage['error_count'] < 0)]

print(f"Rows with impossible negative metrics: {len(negative_counts)}")

# Trial vs Paid Mismatch
paid_trials = df_subscriptions[((df_subscriptions['is_trial'] == 1) & (df_subscriptions['mrr_amount'] > 0)) | 
                               ((df_subscriptions['is_trial'] == 0) & (df_subscriptions['mrr_amount'] <= 0))]
print(f"Rows showing paid transactions during a free trial: {len(paid_trials)}")

Total rows where usage before signup: 13198
Tickets submitted before signup: 1077
Rows with impossible negative metrics: 0
Rows showing paid transactions during a free trial: 0


In [32]:
# Clean feature_usage: Keeping only rows where the usage date is after the signup date
df_feature_usage_clean = usage_check[usage_check['usage_date'] >= usage_check['signup_date']].copy()
df_feature_usage_clean = df_feature_usage_clean.drop(columns=['account_id', 'signup_date'])

# Clean support_tickets: Keeping only rows where submitted_at is after the signup_date
df_support_tickets_clean = ticket_check[ticket_check['submitted_at'] >= ticket_check['signup_date']].copy() 
df_support_tickets_clean = df_support_tickets_clean.drop(columns=['signup_date'])

In [33]:
display(df_feature_usage_clean)
display(df_support_tickets_clean)

,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
1,S-c25263,2023-08-07,feature_5,9,369,0,0
2,S-f29e7f,2023-12-07,feature_3,9,1458,0,0
4,S-f9b1d0,2024-12-02,feature_12,12,900,0,0
5,S-e958ba,2023-10-31,feature_14,11,4103,0,0
10,S-3673f2,2024-08-25,feature_36,9,2061,0,0
...,...,...,...,...,...,...,...
24992,S-38ac0c,2024-12-20,feature_23,6,846,0,0
24995,S-c249fb,2023-07-08,feature_16,7,4116,0,0
24997,S-ad7716,2024-10-03,feature_5,5,2745,0,0
24998,S-dbad62,2024-06-25,feature_5,7,1715,0,0


,account_id,submitted_at,resolution_time_hours,first_response_time_minutes,satisfaction_score
1,A-e43bf7,2024-07-08,27.0,144,NaN
2,A-0f3e88,2024-10-17,19.0,93,4.0
3,A-4c56c9,2024-09-08,47.0,126,5.0
4,A-6f8ad2,2024-11-30,26.0,8,NaN
7,A-72799b,2024-01-29,12.0,56,4.0
...,...,...,...,...,...
1989,A-bb3bd4,2024-12-14,17.0,153,3.0
1992,A-1b707d,2023-09-17,45.0,13,NaN
1993,A-00bed1,2024-05-16,27.0,46,NaN
1995,A-417d2f,2024-11-06,70.0,3,NaN


**Incorrect Data Types**
- The computer initially read all dates as text. I converted them into real calendar dates so I could calculate time-related metrics.

**Duplicates**
- Verified that all tables contain 0 duplicate rows.

**Data Imputation**
- When looking at the customer support table, it is found that out of 2,000 submitted support tickets, only 1,175 had a satisfaction score. The remaining 825 rows were completely blank (NaN).
- A user who completely ignores a satisfaction survey after talking to support is often displaying extreme frustration.
- Therefore, I purposely kept those values as NaN to test a new behavioral hypothesis: <span style="color:lightgreen">Are trial users who ignore support surveys more likely to abandon the product entirely compared to those who take the time to leave feedback?</span> 

**logical inconsistencies**
- Identified a major system tracking issue where 13,198 usage logs and 1,077 support tickets pre-dated user account registration (signup_date).
- Removed all the product usage logs and support tickets that had dates before the user's official signup date. This ensures our analysis only focuses on real activity that happened while the customer was actually using the free trial.

##### _Feature Engineering_

In [34]:
# Create master base table from df_accounts
master_df = df_accounts[['account_id', 'industry', 'signup_date', 'referral_source', 'plan_tier', 'seats']].copy()

# Get trial start date per account
trial_df = df_subscriptions[df_subscriptions['is_trial'] == 1]
trial_start = trial_df.groupby('account_id')['start_date'].min().reset_index()
trial_start.rename(columns={'start_date': 'trial_start_date'}, inplace=True)

# Get paid subscription records
paid_df = df_subscriptions[df_subscriptions['is_trial'] == 0]

# Merge trial start with paid records
conversion_df = paid_df.merge(trial_start, on='account_id', how='inner')

# Calculate days from trial start (not signup)
conversion_df['days_to_convert'] = (
    conversion_df['start_date'] - conversion_df['trial_start_date']
).dt.days

# Filter valid conversions (within trial window)
valid_conversions = conversion_df[
    (conversion_df['days_to_convert'] >= 0) &
    (conversion_df['days_to_convert'] <= 14)
]

# Get converted accounts
converted_accounts = valid_conversions['account_id'].unique()

# Map to master table
master_df['converted_to_paid'] = master_df['account_id'].isin(converted_accounts).astype(int)

print("Target Variable Distribution:")
print(master_df['converted_to_paid'].value_counts())

Target Variable Distribution:
converted_to_paid
0    342
1    158
Name: count, dtype: int64


In [35]:
# Creating metrics for users' early behaviour signals
trial_df = df_subscriptions[df_subscriptions['is_trial'] == 1]
trial_start = trial_df.groupby('account_id')['start_date'].min().reset_index()
trial_start.rename(columns={'start_date': 'trial_start_date'}, inplace=True)

usage = df_feature_usage_clean.merge(
    df_subscriptions[['subscription_id', 'account_id']],
    on='subscription_id',
    how='left'
)
usage = usage.merge(trial_start, on='account_id', how='left')

# Checking behavior strictly to the first 7 days of the trial
usage['days_since_trial'] = (usage['usage_date'] - usage['trial_start_date']).dt.days
first_week_usage = usage[(usage['days_since_trial'] >= 0) & (usage['days_since_trial'] <= 7)]

# Group by account to build our early-behavior columns
usage_features = first_week_usage.groupby('account_id').agg(
    total_clicks=('usage_count', 'sum'),
    total_duration_mins=('usage_duration_secs', lambda x: sum(x) / 60),
    total_errors_encountered=('error_count', 'sum'),
    distinct_features_tried=('feature_name', 'nunique'),
    active_days_count=('usage_date', 'nunique')
).reset_index()

# Merge these features back into our master dataframe (fill missing values with 0 for users who never logged in)
master_df = master_df.merge(usage_features, on='account_id', how='left').fillna({
    'total_clicks': 0, 'total_duration_mins': 0, 'total_errors_encountered': 0, 
    'distinct_features_tried': 0, 'active_days_count': 0
})

In [36]:
trial_df = df_subscriptions[df_subscriptions['is_trial'] == 1]
trial_start = trial_df.groupby('account_id')['start_date'].min().reset_index()
trial_start.rename(columns={'start_date': 'trial_start_date'}, inplace=True)

tickets = df_support_tickets_clean.merge(
    trial_start, on='account_id', how='left'
)

tickets['days_since_trial'] = (tickets['submitted_at'] - tickets['trial_start_date']).dt.days

first_week_tickets = tickets[(tickets['days_since_trial'] >= 0) & (tickets['days_since_trial'] <= 14)]

# Calculate customer service metrics per account
tickets_support = first_week_tickets.groupby('account_id').agg(
    tickets_raised_count=('submitted_at', 'count'),
    avg_response_time_mins=('first_response_time_minutes', 'mean'),
    avg_csat_score=('satisfaction_score', 'mean'),
    skipped_surveys_count=('satisfaction_score', lambda x: x.isna().sum())
).reset_index()

master_df = master_df.merge(tickets_support, on='account_id', how='left')

master_df['tickets_raised_count'] = master_df['tickets_raised_count'].fillna(0)
master_df['skipped_surveys_count'] = master_df['skipped_surveys_count'].fillna(0)
# Use -1 placeholder for accounts that never submitted a ticket
master_df['avg_response_time_mins'] = master_df['avg_response_time_mins'].fillna(-1) 
master_df['avg_csat_score'] = master_df['avg_csat_score'].fillna(-1)

In [37]:
display(master_df)

,account_id,industry,signup_date,referral_source,plan_tier,seats,converted_to_paid,total_clicks,total_duration_mins,total_errors_encountered,distinct_features_tried,active_days_count,tickets_raised_count,avg_response_time_mins,avg_csat_score,skipped_surveys_count
0,A-2e4581,EdTech,2024-10-16,partner,Basic,9,1,7.0,34.066667,0.0,1.0,1.0,0.0,-1.0,-1.0,0.0
1,A-43a9e3,FinTech,2023-08-17,other,Basic,18,0,0.0,0.000000,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0
2,A-0a282f,DevTools,2024-08-27,organic,Basic,1,1,0.0,0.000000,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0
3,A-1f0ac7,HealthTech,2023-08-27,other,Basic,24,0,9.0,42.750000,3.0,1.0,1.0,0.0,-1.0,-1.0,0.0
4,A-ce550d,HealthTech,2024-10-27,event,Enterprise,35,1,34.0,222.883333,3.0,4.0,3.0,0.0,-1.0,-1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,A-8ae3fc,DevTools,2024-06-28,ads,Pro,9,0,11.0,66.183333,0.0,1.0,1.0,0.0,-1.0,-1.0,0.0
496,A-55f257,FinTech,2023-12-21,organic,Basic,9,1,0.0,0.000000,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0
497,A-d26ab4,DevTools,2024-11-07,organic,Basic,9,1,25.0,20.683333,0.0,2.0,2.0,0.0,-1.0,-1.0,0.0
498,A-712533,EdTech,2023-07-31,organic,Pro,18,0,0.0,0.000000,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0


**Features**
- `total_clicks`: Tracks the total number of actions a user takes in their first week. 
> *It helps us see if active users are more likely to buy than users who log in once and click around aimlessly.*
- `total_duration_mins`: The total time (in minutes) spent inside the app during the first week.
> *It helps separate users who just open the app and close it from those who are genuinely spending time working inside it.*
- `total_errors_encountered`: The total number of system errors or software bugs the user hit in their first 7 days.
> *This lets us calculate exactly how many technical bugs a user will tolerate before they lose patience, give up, and abandon the trial.*
- `distinct_features_tried`: Out of the 40 tools available, this counts how many different features they experimented with.
> *It helps us understand if users need to explore the whole platform to see its value, or if they only need one specific feature to be satisfied.*
- `active_days_count`: How many separate days out of the first 7 they actually opened the tool and used it.
> *Logging in 4 days out of 7 shows a routine, whereas logging in for only 1 day indicates a lack of consistent interest.*
- `tickets_raised_count`: The number of times they asked customer support for help during their 14-day trial.
> *Measures user confusion or blockers.*
- `avg_response_time_mins`: How many minutes it took our team to reply to their questions.
> *It allows us to prove if slow support directly causes users to lose momentum and abandon the trial.*
- `avg_csat_score`: The star rating (1–5) they gave our support team after their issue was resolved.
> *If they leave a 1-star review, we can flag them as an immediate risk of leaving.*
- `skipped_surveys_count`: The number of times they opened a support ticket but completely ignored the feedback survey at the end.
> *It tests whether users who completely ignore communication are silently "ghosting" the app and heading toward conversion failure.*
- `converted_to_paid`: A "1" means they entered their credit card and upgraded to a paid account. A "0" means their trial expired and they walked away.
> *Every single feature above is analyzed specifically to see how it affects this one number.*

In [38]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   account_id                500 non-null    object        
 1   industry                  500 non-null    object        
 2   signup_date               500 non-null    datetime64[ns]
 3   referral_source           500 non-null    object        
 4   plan_tier                 500 non-null    object        
 5   seats                     500 non-null    int64         
 6   converted_to_paid         500 non-null    int64         
 7   total_clicks              500 non-null    float64       
 8   total_duration_mins       500 non-null    float64       
 9   total_errors_encountered  500 non-null    float64       
 10  distinct_features_tried   500 non-null    float64       
 11  active_days_count         500 non-null    float64       
 12  tickets_raised_count  

##### _Loading Into SQL_

In [40]:
print("Uploading tables to MySQL...")

df_accounts.to_sql('accounts', con=engine, if_exists='append', index=False)
df_subscriptions.to_sql('subscriptions', con=engine, if_exists='append', index=False)
df_feature_usage_clean.to_sql('feature_usage', con=engine, if_exists='append', index=False)
df_support_tickets_clean.to_sql('support_tickets', con=engine, if_exists='append', index=False)
master_df.to_sql('master_analytics', con=engine, if_exists='append', index=False)

print("All tables successfully loaded into MySQL")

Uploading tables to MySQL...
All tables successfully loaded into MySQL
